We will analyze the MNIST dataset. We will compare the accuracies for applying PCA (linear) versus kernel PCA (non linear) to test the linearity of the dataset

This dataset is suitable for classification rather than for clustering as it has labels. Therefore we will compare the accuracies based on K-NN (k=10) classification model.

We will use Nyström insead of KernelPCA from sklearn. Nyström uses a subset of the data to speed up the kernel computation time and reduce memory usage.

In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.kernel_approximation import Nystroem
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix
from scipy.stats import mode
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

### Fetch MNIST data

In [2]:
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
y = y.astype(int)
X = X / 255.0

### Split the data into training and test sets

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

First, we will apply K-NN without PCA to see how it performs without dimensionality reduction

In [9]:
n_neighbors=10
knn = KNeighborsClassifier(n_neighbors=n_neighbors)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"K-NN Accuracy without PCA: {accuracy:.4f}")

K-NN Accuracy without PCA: 0.9658


### Apply PCA for dimensionality reduction and train K-NN classifier

In [4]:
n_components = 50
pca = PCA(n_components=n_components, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)


knn_pca = KNeighborsClassifier(n_neighbors=n_neighbors)
knn_pca.fit(X_train_pca, y_train)

y_pred_pca = knn_pca.predict(X_test_pca)
accuracy_pca = accuracy_score(y_test, y_pred_pca)
print(f"K-NN Accuracy with PCA: {accuracy_pca:.4f}")

K-NN Accuracy with PCA: 0.9734


### Apply Kernel PCA for dimensionality reduction and train K-NN classifier

In [7]:
kpca = Nystroem(n_components=n_components, kernel="rbf", gamma=0.04, random_state=42)
X_train_kpca = kpca.fit_transform(X_train)
X_test_kpca = kpca.transform(X_test)

knn_kpca = KNeighborsClassifier(n_neighbors=n_neighbors)
knn_kpca.fit(X_train_kpca, y_train)

y_pred_kpca = knn_kpca.predict(X_test_kpca)
accuracy_kpca = accuracy_score(y_test, y_pred_kpca)
print(f"K-NN Accuracy with Kernel PCA: {accuracy_kpca:.4f}")

K-NN Accuracy with Kernel PCA: 0.9197


In [10]:
print("\nComparison:")
print(f"K-NN Accuracy without Dimensionality Redcution: {accuracy:.4f}")
print(f"K-NN Accuracy with PCA: {accuracy_pca:.4f}")
print(f"K-NN Accuracy with Kernel PCA: {accuracy_kpca:.4f}")


Comparison:
K-NN Accuracy without Dimensionality Redcution: 0.9658
K-NN Accuracy with PCA: 0.9734
K-NN Accuracy with Kernel PCA: 0.9197


First of all, dimensionality redcution indeed improved classification results (0.9734 vs. 0.9658). Secondly, The regular PCA performed better than the kernel PCA suggesting that the data is probably linear and no kernel transformation is needed.